XOM vs CVX

In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler


In [ ]:
file_cvx_path = '/Users/wesleylu/Desktop/CS4641/linear_regression/cvx.us.txt'
file_xom_path = '/Users/wesleylu/Desktop/CS4641/linear_regression/xom.us.txt'

xom_data = pd.read_csv(file_xom_path)
cvx_data = pd.read_csv(file_cvx_path)

xom_data['Log_Close'] = np.log(xom_data['Close'])
cvx_data['Log_Close'] = np.log(cvx_data['Close'])

xom_output_path = 'xom_with_log.csv'
cvx_output_path = 'cvx_with_log.csv'

xom_data.to_csv(xom_output_path, index=False)
cvx_data.to_csv(cvx_output_path, index=False)

print(f"XOM file with log-transformed prices saved to: {xom_output_path}")
print(f"CVX file with log-transformed prices saved to: {cvx_output_path}")


XOM file with log-transformed prices saved to: xom_with_log.csv
CVX file with log-transformed prices saved to: cvx_with_log.csv


In [ ]:
# Load the log-transformed data for both stocks
stock_a = pd.read_csv("xom_with_log.csv")
stock_b = pd.read_csv("cvx_with_log.csv")

# filter out the columns not needed
stock_a = stock_a[['Date', 'Log_Close']].rename(columns={'Log_Close': 'Log_A'})
stock_b = stock_b[['Date', 'Log_Close']].rename(columns={'Log_Close': 'Log_B'})

# merge the data on date
data = pd.merge(stock_a, stock_b, on='Date')

# prepare for linear regression
X = data['Log_A'].values.reshape(-1, 1)  # Independent variable (Log prices of stock A)
y = data['Log_B'].values  # Dependent variable (Log prices of stock B)

# train the model
model = LinearRegression()
model.fit(X, y)

# Extract the hedge ratio and intercept
hedge_ratio = model.coef_[0]
intercept = model.intercept_

print(f"Hedge Ratio: {hedge_ratio}, Intercept: {intercept}")

# Calculate the spread
data['Spread'] = model.predict(X) - data['Log_B']

# normalize spread (z-score)
scaler = StandardScaler()
data['Z-Score'] = scaler.fit_transform(data['Spread'].values.reshape(-1, 1))

# generate trading signals based on z-score threshold 
threshold = 1.0  # Define the z-score threshold for signals
data['Signal'] = np.where(data['Z-Score'] < -threshold, 'Long',
                          np.where(data['Z-Score'] > threshold, 'Short', 'Hold'))


print(data[['Date', 'Log_A', 'Log_B', 'Spread', 'Z-Score', 'Signal']].head())
data.to_csv("trading_signals.csv", index=False)

Hedge Ratio: 1.1552370224700717, Intercept: -0.5197285432414263
         Date     Log_A     Log_B    Spread   Z-Score Signal
0  1970-01-02  0.403597 -0.282628  0.229150  1.276453  Short
1  1970-01-05  0.425072 -0.272203  0.243532  1.356572  Short
2  1970-01-06  0.419762 -0.293030  0.258226  1.438422  Short
3  1970-01-07  0.414623 -0.293030  0.252289  1.405350  Short
4  1970-01-08  0.414623 -0.272203  0.231462  1.289334  Short
Trading signals saved to 'trading_signals.csv'
